# 06 - Inference & Submission

In [ ]:
import os, sys, zipfile
from pathlib import Path

# Cari root repo: di laptop = folder di atas notebooks/, di Kaggle = dataset mlc-src
if os.path.exists("/kaggle/input"):
    hits = list(Path("/kaggle/input").rglob("src/data.py"))
    if hits:
        ROOT = hits[0].parent.parent
    else:  # dataset masih berupa zip
        ROOT = Path("/kaggle/working/_repo")
        for z in Path("/kaggle/input").rglob("*.zip"):
            if z.stem in ("src", "configs", "annotations"):
                zipfile.ZipFile(z).extractall(ROOT / z.stem)
else:
    ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

from src.data import ON_KAGGLE, get_path, load_config
from src.seed import seed_everything

cfg = load_config()
seed_everything(cfg["SEED"])
print("Kaggle:", ON_KAGGLE, "| data:", get_path(cfg, "raw"))

## Muat threshold dan bobot hasil notebook 05

Notebook ini **hanya inferensi**: tidak ada training, tidak ada statistik apa pun dari data test,
dan tiap sequence test diprediksi sendiri dari 30 frame-nya. Threshold dibaca apa adanya dari
`models/threshold.json` (hasil OOF train di notebook 05).

In [ ]:
import json

import numpy as np
import pandas as pd
import torch

from src.data import get_path
from src.models import ModelTemporal, jumlah_parameter, muat_test_mentah, prediksi_batch

METRIK = cfg["METRIC"]
SEQ_LEN = cfg["SEQ_LEN"]
N_SEED = 3                                  # notebook 05: 5 fold x 3 seed
N_MODEL_HARAP = cfg["N_FOLDS"] * N_SEED     # = 15
AMP = torch.cuda.is_available()             # mixed precision hanya di cuda
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

dir_model = get_path(cfg, "models", read=True)
dir_fitur = get_path(cfg, "features", read=True)
print("bobot dari :", dir_model)
print("fitur dari :", dir_fitur)
print("device:", device, "| mixed precision:", AMP, "| metrik lomba:", METRIK)

p_thr = dir_model / "threshold.json"
if not p_thr.exists():
    raise SystemExit(
        f"threshold.json tidak ada di {dir_model}.\n"
        "Jalankan notebook 05 lebih dulu, lalu pasang output kernel 05 lewat kernel_sources "
        "di kernel-metadata.json notebook 06 (scripts/kaggle_run.py push 06)."
    )
info_thr = json.loads(p_thr.read_text(encoding="utf-8"))
THR = float(info_thr["threshold"])
if info_thr.get("metrik") != METRIK:
    print(f"PERINGATAN: threshold.json dibuat untuk metrik {info_thr.get('metrik')!r}, "
          f"sedangkan configs/config.yaml memakai {METRIK!r}.")
print("threshold.json:", info_thr)

# Hanya bobot asli notebook 05; berkas uji cepat (_smoke) dan berkas lain diabaikan.
bobot = [p for p in sorted(dir_model.glob("temporal_f*_s*.pt")) if "_smoke" not in p.name]
print("file bobot ditemukan:", len(bobot))
for p in bobot:
    print("  ", p.name)
if len(bobot) != N_MODEL_HARAP:
    raise SystemExit(
        f"Ditemukan {len(bobot)} file bobot temporal_f*_s*.pt di {dir_model}, "
        f"seharusnya {N_MODEL_HARAP} (5 fold x 3 seed).\n"
        "Notebook 05 mungkin belum selesai atau output kernelnya belum dipasang. "
        "Jalankan ulang notebook 05 (model yang sudah ada akan dilewati), lalu ulangi notebook 06."
    )

## Prediksi test: rata-rata probabilitas semua model

In [ ]:
# emb_test.npy [1515, 30, 768] -> tiap baris satu sequence, diproses sendiri-sendiri.
# Tidak ada hubungan antar-sequence dan nomor id test tidak dipakai sebagai fitur.
Xte, ids_te = muat_test_mentah(dir_fitur, seq_len=SEQ_LEN)
print("fitur test:", Xte.shape, "| jumlah id:", len(ids_te))

prob_per_model, n_muat = {}, 0
for p in bobot:
    try:
        isi = torch.load(p, map_location=device, weights_only=True)
    except Exception:                       # isi bukan tensor murni (metadata dict)
        isi = torch.load(p, map_location=device, weights_only=False)
    dim = int(isi.get("dim_fitur", Xte.shape[2]))
    seq = int(isi.get("seq_len", SEQ_LEN))
    n_perilaku = len(isi.get("peta_perilaku") or {})
    if dim != Xte.shape[2] or seq != SEQ_LEN:
        raise SystemExit(f"{p.name}: dilatih untuk dim {dim} seq {seq}, "
                         f"fitur test dim {Xte.shape[2]} seq {SEQ_LEN}.")
    model = ModelTemporal(dim_fitur=dim, seq_len=seq, n_perilaku=n_perilaku).to(device)
    sisa = model.load_state_dict(isi["state_dict"], strict=True)      # strict: tidak boleh ada kunci hilang
    assert not sisa.missing_keys and not sisa.unexpected_keys, (p.name, sisa)
    model.eval()                            # prediksi_batch juga memakai eval() + torch.no_grad()
    with torch.no_grad():
        prob_per_model[p.name] = prediksi_batch(model, Xte, device, amp=AMP)
    n_muat += 1
    del model
    if device.type == "cuda":
        torch.cuda.empty_cache()

print("model dimuat:", n_muat, "dari", N_MODEL_HARAP, "| parameter per model:",
      f"{jumlah_parameter(ModelTemporal(dim_fitur=int(Xte.shape[2]), seq_len=SEQ_LEN, n_perilaku=n_perilaku)):,}")
assert n_muat == N_MODEL_HARAP, f"baru {n_muat} model yang dimuat"

prob_te = np.mean(np.stack([prob_per_model[k] for k in sorted(prob_per_model)]), axis=0)
assert prob_te.shape == (len(ids_te),) and np.isfinite(prob_te).all()
print("prob rata-rata: min %.4f | median %.4f | maks %.4f" % (
    prob_te.min(), float(np.median(prob_te)), prob_te.max()))
print("threshold dipakai:", THR, "| perkiraan rasio label 1:", round(float((prob_te >= THR).mean()), 4))

## submission.csv + validasi format

In [ ]:
pred = pd.DataFrame({"id": ids_te, "label": (prob_te >= THR).astype(int)})

contoh = pd.read_csv(get_path(cfg, "raw") / "sample_submission.csv")


def kunci(s):
    """Cocokkan id lewat teks, supaya id angka maupun teks sama-sama jalan.
    Kolom id yang ditulis tetap milik sample_submission (urutan dan dtype aslinya)."""
    return s.astype(str).str.strip()


sub = contoh[["id"]].copy()
sub["_k"] = kunci(sub["id"])
sub = sub.merge(pred.assign(_k=kunci(pred["id"]))[["_k", "label"]], on="_k", how="left").drop(columns="_k")

assert sub["label"].notna().all(), (
    "ada id di sample_submission yang tidak punya prediksi | contoh id sample=%r vs emb_test=%r"
    % (contoh["id"].iloc[0], ids_te[0]))
sub["label"] = sub["label"].astype(int)
assert len(sub) == 1515, "jumlah baris %d, seharusnya 1515" % len(sub)
assert sub["id"].tolist() == contoh["id"].tolist(), "urutan id tidak sama dengan sample_submission"
assert list(sub.columns) == ["id", "label"], sub.columns.tolist()
assert set(sub["label"].unique()) <= {0, 1}, "label bukan 0/1"
assert sub["label"].dtype.kind == "i", "label bukan integer"

p_sub = get_path(cfg, "submissions") / "submission.csv"
p_sub.parent.mkdir(parents=True, exist_ok=True)
sub.to_csv(p_sub, index=False)
print("Tersimpan:", p_sub, "|", len(sub), "baris | rasio label 1:", round(float(sub["label"].mean()), 4))
print(sub.head(3).to_string(index=False))

## Bandingkan dengan submission_05.csv (kalau ada)

In [ ]:
# submission_05.csv dibuat di dalam notebook 05 dengan bobot dan threshold yang sama,
# jadi labelnya harus sama persis. Kalau file tidak ada, langkah ini cukup dilewati.
kandidat = [get_path(cfg, "submissions") / "submission_05.csv"]
if ON_KAGGLE:
    kandidat += sorted(Path("/kaggle/input").rglob("submission_05.csv"))
p_lama = next((p for p in kandidat if p.exists()), None)

if p_lama is None:
    print("submission_05.csv tidak ditemukan - perbandingan dilewati (bukan kegagalan).")
else:
    lama = pd.read_csv(p_lama)
    gabung = sub.assign(_k=kunci(sub["id"])).merge(
        lama.assign(_k=kunci(lama["id"]))[["_k", "label"]], on="_k", suffixes=("", "_05"))
    n_beda = int((gabung["label"] != gabung["label_05"]).sum())
    print("dibandingkan dengan:", p_lama, "|", len(gabung), "id cocok | label berbeda:", n_beda)
    if len(gabung) != len(sub):
        print(f"PERINGATAN: hanya {len(gabung)} dari {len(sub)} id yang cocok dengan submission_05.csv.")
    if n_beda:
        print(f"PERINGATAN: {n_beda} label berbeda dari submission_05.csv "
              "(seharusnya 0 karena bobot dan threshold sama). "
              "Cek apakah bobot/threshold yang dipasang benar-benar dari kernel 05 yang sama.")
    else:
        print("OK: label sama persis dengan submission_05.csv.")

## Daftar file yang dikumpulkan

In [ ]:
def lapor(judul, daftar):
    ada = [p for p in daftar if p.exists()]
    hilang = [p for p in daftar if not p.exists()]
    ukuran = sum(p.stat().st_size for p in ada if p.is_file()) / 1e6
    print(f"{judul:28s} {len(ada):3d} ada" + (f", {len(hilang)} tidak ada" if hilang else "")
          + (f"  ({ukuran:.1f} MB)" if ukuran else ""))
    for p in ada[:40]:
        print("   ", p.name if p.is_file() else p.name + "/")
    for p in hilang:
        print("    TIDAK ADA:", p)


dir_nb = ROOT / "notebooks"
lapor("notebooks 01-06", [dir_nb / n for n in sorted(
    p.name for p in dir_nb.glob("0[1-6]_*.ipynb"))] if dir_nb.exists() else [dir_nb])
lapor("src/ (modul)", sorted((ROOT / "src").glob("*.py")))
lapor("configs/", sorted((ROOT / "configs").glob("*.yaml")))
lapor("annotations/", sorted(p for p in (ROOT / "annotations").iterdir() if p.is_file()))
lapor("models/*.pt", bobot)
lapor("models/threshold.json", [p_thr])
lapor("submission", [p_sub])
print("\nCatatan: di Kaggle folder notebooks/ tidak ikut dataset src; notebook 01-06 dikirim "
      "sebagai kernel lewat scripts/kaggle_run.py.")